# 03 — Gold Layer Modeling & Business Metrics

**Medallion Tier 3.** Build a **Star Schema** (SCD-managed dimensions + an order-item fact), then the
two required business marts.

Professional touches:
- **SCD Type 2** for `dim_customer` & `dim_product` (full history: `valid_from/valid_to/is_current`).
- **SCD Type 1** for `dim_location`, `dim_payment_method`; **generated** `dim_date` calendar.
- **Surrogate keys** + an **Unknown member (-1)** so fact→dim joins never drop rows.
- **Idempotent fact MERGE** keyed on `(order_id, line_number)` + partitioning + `OPTIMIZE/ZORDER`.
- **Marts** apply the *Lọc dữ liệu ảo* filter (`order_status<>'Failed'` AND `feedback_is_valid`).

In [ ]:
batch_id = ""

In [ ]:
%run 00_common_utils

In [ ]:
batch_id = batch_id or new_batch_id()
G = CONFIG["gold"]
DIM = G["dimensions"]
log(f"Gold batch_id = {batch_id}")

## 1. Dimensions

In [ ]:
run = start_run("gold", "dimensions", batch_id)
try:
    sales = spark.table("silver_sales")
    items = (sales.select("order_id", F.posexplode("items").alias("line_number", "item"))
                  .select("order_id", "line_number",
                          F.col("item.product_id").alias("product_id"),
                          F.col("item.category").alias("category")))

    # dim_date (generated calendar)
    d = DIM["dim_date"]
    merge_upsert(generate_calendar(d["start_date"], d["end_date"]), d["table"], ["date_key"])

    # dim_customer (SCD2)
    c = DIM["dim_customer"]
    cust_src = (sales.select("customer_email", "customer_name", "customer_phone", "customer_age")
                     .where(F.col("customer_email").isNotNull()))
    scd2_upsert(cust_src, c["table"], c["surrogate_key"], c["business_key"], c["tracked_columns"], batch_id)

    # dim_product (SCD2)
    p = DIM["dim_product"]
    scd2_upsert(items.select("product_id", "category"), p["table"], p["surrogate_key"],
                p["business_key"], p["tracked_columns"], batch_id)

    # dim_location (SCD1)
    l = DIM["dim_location"]
    scd1_upsert(sales.select("location").where(F.col("location").isNotNull()).distinct(),
                l["table"], l["surrogate_key"], l["business_key"], l["tracked_columns"], batch_id)

    # dim_payment_method (SCD1)
    pm = DIM["dim_payment_method"]
    scd1_upsert(sales.select("payment_method").where(F.col("payment_method").isNotNull()).distinct(),
                pm["table"], pm["surrogate_key"], pm["business_key"], pm["tracked_columns"], batch_id)

    # dim_exchange_rate (reference)
    fx = (spark.table("silver_exchange_rate")
          .select("year", "month", "from_currency", "to_currency", "exchange_rate")
          .withColumn("rate_key", F.concat_ws("-", "year", "month", "from_currency", "to_currency")))
    merge_upsert(fx, DIM["dim_exchange_rate"]["table"], ["rate_key"])

    for name in [d["table"], c["table"], p["table"], l["table"], pm["table"], DIM["dim_exchange_rate"]["table"]]:
        print(f"{name}: {spark.table(name).count()}")
    end_run(run, "SUCCESS", message="dimensions built")
except Exception as e:
    end_run(run, "FAILED", message=str(e)); raise

## 2. fact_sales — explode to item grain, convert to VND, attach surrogate keys

In [ ]:
run = start_run("gold", "fact_sales", batch_id)
try:
    UNK = G["fact"]["fact_sales"]["unknown_member_key"]
    rates = spark.table("silver_exchange_rate")
    sales = spark.table("silver_sales")

    line = (sales.select(
                "order_id", "order_date", F.col("order_year").alias("year"), F.col("order_month").alias("month"),
                "customer_email", "location", "payment_method", "device_type", "order_status",
                "discount_code", "has_discount", "feedback_score", "feedback_is_valid",
                F.posexplode("items").alias("line_number", "item"))
            .select("*",
                    F.col("item.product_id").alias("product_id"),
                    F.col("item.category").alias("category"),
                    F.col("item.price").alias("unit_price_usd"),
                    F.col("item.quantity").alias("quantity"))
            .drop("item")
            .withColumn("line_amount_usd", F.col("unit_price_usd") * F.col("quantity")))

    # current dimension versions for surrogate-key lookups
    dc = spark.table("dim_customer").where("is_current = true").select("customer_email", "customer_sk")
    dp = spark.table("dim_product").where("is_current = true").select("product_id", "product_sk")
    dl = spark.table("dim_location").select("location", "location_sk")
    dpm = spark.table("dim_payment_method").select("payment_method", "payment_method_sk")

    fact = (line
        .join(rates, ["year", "month"], "left")
        .join(dc, "customer_email", "left").join(dp, "product_id", "left")
        .join(dl, "location", "left").join(dpm, "payment_method", "left")
        .withColumn("line_amount_vnd", F.col("line_amount_usd") * F.col("exchange_rate"))
        .withColumn("date_key", F.date_format("order_date", "yyyyMMdd").cast("int"))
        .select(
            "order_id", "line_number", "date_key",
            F.coalesce("customer_sk", F.lit(UNK)).alias("customer_sk"),
            F.coalesce("product_sk", F.lit(UNK)).alias("product_sk"),
            F.coalesce("location_sk", F.lit(UNK)).alias("location_sk"),
            F.coalesce("payment_method_sk", F.lit(UNK)).alias("payment_method_sk"),
            "year", "month", "category", "location", "payment_method", "device_type",
            "order_status", "discount_code", "has_discount", "feedback_score", "feedback_is_valid",
            "quantity", "unit_price_usd", "line_amount_usd", "exchange_rate", "line_amount_vnd")
        .withColumn("_batch_id", F.lit(batch_id))
        .withColumn("_gold_loaded_at", F.current_timestamp()))

    merge_upsert(fact, G["fact"]["fact_sales"]["table"], G["fact"]["fact_sales"]["business_key"],
                 partition_by=G["fact"]["fact_sales"]["partition_by"])
    n = spark.table("fact_sales").count()
    miss = spark.table("fact_sales").where("exchange_rate IS NULL").count()
    print(f"fact_sales rows: {n}, rows missing exchange rate: {miss}")
    end_run(run, "SUCCESS", rows_written=n, message=f"missing_fx={miss}")
except Exception as e:
    end_run(run, "FAILED", message=str(e)); raise

## 3. Business marts (Lọc dữ liệu ảo filter applied)

In [ ]:
run = start_run("gold", "marts", batch_id)
try:
    reported = spark.table("fact_sales").where(G["report_filter"]["expression"])

    # Mart 1 — monthly Total_Revenue_VND by category
    rev = (reported.groupBy("year", "month", "category")
        .agg(F.round(F.sum("line_amount_vnd"), 0).alias("total_revenue_vnd"),
             F.round(F.sum("line_amount_usd"), 2).alias("total_revenue_usd"),
             F.countDistinct("order_id").alias("order_count"),
             F.sum("quantity").alias("units_sold"))
        .orderBy("year", "month", "category")
        .withColumn("_batch_id", F.lit(batch_id)))
    rev.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(G["marts"]["monthly_revenue_vnd_by_category"])

    # Mart 2 — promo effectiveness by region (order grain)
    order_grain = reported.select("order_id", "location", "has_discount").dropDuplicates(["order_id"])
    promo = (order_grain.groupBy("location")
        .agg(F.count("order_id").alias("total_orders"),
             F.sum(F.col("has_discount").cast("int")).alias("orders_with_discount"))
        .withColumn("pct_orders_with_discount",
                    F.round(F.col("orders_with_discount") / F.col("total_orders") * 100, 2))
        .orderBy(F.desc("total_orders"))
        .withColumn("_batch_id", F.lit(batch_id)))
    promo.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(G["marts"]["promo_effectiveness_by_region"])

    print("=== Mart 1: monthly revenue by category ==="); rev.show(10, truncate=False)
    print("=== Mart 2: promo effectiveness by region ==="); promo.show(10, truncate=False)
    end_run(run, "SUCCESS", message="marts built")
except Exception as e:
    end_run(run, "FAILED", message=str(e)); raise

## 4. Table maintenance — OPTIMIZE / ZORDER / VACUUM

In [ ]:
mnt = G["maintenance"]
if mnt.get("optimize"):
    optimize_table("fact_sales", mnt.get("zorder_fact"))
    for t in ["dim_customer", "dim_product", "dim_location", "dim_date"]:
        optimize_table(t)
    vacuum_table("fact_sales", mnt.get("vacuum_retain_hours", 168))

## 5. Sanity totals

In [ ]:
print("Total_Revenue_VND by category (all months, reported):")
(spark.table(G["marts"]["monthly_revenue_vnd_by_category"])
    .groupBy("category").agg(F.sum("total_revenue_vnd").alias("total_revenue_vnd"))
    .orderBy(F.desc("total_revenue_vnd")).show(truncate=False))
print("SCD2 customer history sample:")
spark.table("dim_customer").select("customer_sk", "customer_email", "is_current", "valid_from", "valid_to").show(5, truncate=False)